# Rag pipeline for document loading and preprocessing

In [12]:
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [13]:
### Read all types of documents using document loaders.
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: DataMining(1)_.pdf
  ✓ Loaded 786 pages

Total documents loaded: 786


In [29]:
#text splitting into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, 
    chunk_overlap=chunk_overlap,
    length_function=len,
    separators=["\n\n", "\n", " ", ""])
    split_docs=text_splitter.split_documents(documents)
    print(f"split{len(documents)} documents into {len(split_docs)} chunks")
    #show example of chunks
    if split_docs:
        print(f"\nExample chunk:\n{split_docs[0].page_content[:500]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs 

In [30]:
chunks=split_documents(all_pdf_documents)

split786 documents into 3081 chunks

Example chunk:
Data Mining
Concepts and Techniques...
Metadata: {'producer': 'Acrobat Distiller 9.5.5 (Windows)', 'creator': 'Elsevier', 'creationdate': '2022-07-01T15:33:03+03:00', 'author': 'Han, Jiawei', 'elsevierbookpdfspecifications': '1.3', 'elsevierwebpdfspecifications': '7.0', 'moddate': '2022-07-01T15:34:19+03:00', 'subject': 'Data Mining: Concepts and Techniques, Fourth edition (2023) 784pp. 978-0-12-811760-6', 'title': 'Data Mining', 'robots': 'noindex', 'ebx_publisher': 'Elsevier Science', 'source': '../data/pdf/DataMining(1)_.pdf', 'total_pages': 786, 'page': 1, 'page_label': 'i', 'source_file': 'DataMining(1)_.pdf', 'file_type': 'pdf'}


# Embedding and Vector store DB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_55765/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


# Vector Store

In [33]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [34]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 9.5.5 (Windows)', 'creator': 'Elsevier', 'creationdate': '2022-07-01T15:33:03+03:00', 'author': 'Han, Jiawei', 'elsevierbookpdfspecifications': '1.3', 'elsevierwebpdfspecifications': '7.0', 'moddate': '2022-07-01T15:34:19+03:00', 'subject': 'Data Mining: Concepts and Techniques, Fourth edition (2023) 784pp. 978-0-12-811760-6', 'title': 'Data Mining', 'robots': 'noindex', 'ebx_publisher': 'Elsevier Science', 'source': '../data/pdf/DataMining(1)_.pdf', 'total_pages': 786, 'page': 1, 'page_label': 'i', 'source_file': 'DataMining(1)_.pdf', 'file_type': 'pdf'}, page_content='Data Mining\nConcepts and Techniques'),
 Document(metadata={'producer': 'Acrobat Distiller 9.5.5 (Windows)', 'creator': 'Elsevier', 'creationdate': '2022-07-01T15:33:03+03:00', 'author': 'Han, Jiawei', 'elsevierbookpdfspecifications': '1.3', 'elsevierwebpdfspecifications': '7.0', 'moddate': '2022-07-01T15:34:19+03:00', 'subject': 'Data Mining: Concepts and Techniques, F

In [35]:
### convert text to embeddings
texts=[doc.page_content for doc in chunks]
### Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)
#store in vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 3081 texts...


Batches:   0%|          | 0/97 [00:00<?, ?it/s]

Generated embeddings with shape: (3081, 384)
Adding 3081 documents to vector store...
Successfully added 3081 documents to vector store
Total documents in collection: 3081


In [38]:
class RagRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager 
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RagRetriever(vectorstore,embedding_manager) 

In [39]:
rag_retriever

In [47]:
rag_retriever.retrieve("ata mining: an essential step in knowledge discovery")

Retrieving documents for query: 'ata mining: an essential step in knowledge discovery'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_23cdd9a8_89',
  'content': 'About the authors\nJiawei Han is a Michael Aiken Chair Professor in the Department of Computer Science at the Uni-\nversity of Illinois at Urbana-Champaign. He has received numerous awards for his contributions on\nresearch into knowledge discovery and data mining, including ACM SIGKDD Innovation Award\n(2004), IEEE Computer Society Technical Achievement Award (2005), and IEEE W. Wallace McDow-\nell Award (2009). He is a Fellow of ACM and a Fellow of IEEE. He served as founding Editor-in-Chief\nof ACM Transactions on Knowledge Discovery from Data (2006–2011) and as an editorial board mem-\nber of several journals, including IEEE Transactions on Knowledge and Data Engineering and Data\nMining and Knowledge Discovery .\nJian Pei is currently Professor of Computer Science, Biostatistics and Bioinformatics, and Electrical\nand Computer Engineering at Duke University. He received a Ph.D. degree in computing science from',
  'metadata': {'source': '..

In [63]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

key = os.getenv("GROQ_API_KEY")
### initialize the groq llm
groq_api_key=os.getenv("GROQ_API_KEY")
llm=chat_groq=ChatGroq(groq_api_key=key,model_name="openai/gpt-oss-120b", temperature=0.4, max_tokens=1024)
### Simple RAG function
def rag_simple(query,retriever,llm,top_k=3):
    # Step 1: Retrieve relevant documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    
    if not retrieved_docs:
        return "No relevant documents found."
    
    # Step 2: Prepare context for LLM
    context = "\n\n".join([f"Document {doc['rank']} (Score: {doc['similarity_score']:.4f}):\n{doc['content']}" for doc in retrieved_docs])
    if not context:
        return "No relevant content found in retrieved documents."
    # Step 3: Generate response using LLM
    prompt = f"Answer the following question based on the provided documents:\n\nQuestion: {query}\n\nContext:\n{context}\n\nAnswer:"
    
    try:
        response = llm.invoke(prompt.format(query=query, context=context))
        return response.content
    except Exception as e:
        print(f"Error generating response: {e}")
        return "An error occurred while generating the response."

In [64]:
answer=rag_simple("what is pattern mining? what are the various kinds of this", rag_retriever, llm)
answer

Retrieving documents for query: 'what is pattern mining? what are the various kinds of this'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


'**Pattern mining** is a data‑mining activity that looks for regularities, regular structures or “patterns” that appear in a data set often enough (or satisfy some other significance criterion) to be considered useful knowledge.  \nA pattern can be any construct that can be matched against the data – a set of items that co‑occur, an ordered list of events, a subgraph, a subtree, etc.  The goal is to discover those constructs automatically, so that they can be examined, interpreted, or fed into downstream tasks such as classification, clustering, recommendation, or decision‑making.\n\n---\n\n### Main kinds of pattern mining (as reflected in the supplied documents)\n\n| Kind | What it looks for | Typical data representation | Example |\n|------|-------------------|-----------------------------|---------|\n| **Frequent‑itemset mining** | Sets of items that appear together in many transactions. | Transaction tables (binary or count). | `{milk, bread}` bought together in >\u202f30\u202f% of

# Enhance the pipeline features

In [93]:
def rag_advanced(query,retriever,llm,top_k=4,min_score=0.2,return_context=False):
    # Step 1: Retrieve relevant documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    
    if not retrieved_docs:
        return "No relevant documents found."
    
    # Step 2: Prepare context for LLM
    context = "\n\n".join([f"Document {doc['rank']} (Score: {doc['similarity_score']:.4f}):\n{doc['content']}" for doc in retrieved_docs])
    if not context:
        return "No relevant content found in retrieved documents."
    sources=[{'source': doc['metadata'].get('source_file',doc['metadata'].get('source', 'Unknown'))
    , 'page': doc['metadata'].get('page', 'N/A'),
         'preview': doc['content'][:200]+ '...'
    , 'score': doc['similarity_score']} for doc in retrieved_docs]

    confidence=max(doc['similarity_score'] for doc in retrieved_docs)
    
    # Step 3: Generate response using LLM
    prompt = f"""You are a document-grounded assistant.Answer ONLY using information present in the provided context.
    If the answer is not explicitly stated or cannot be reasonably inferred from the context, say:The provided documents do not contain sufficient information to answer this question.Do not use external knowledge.:\n\nQuestion: {query}\n\nContext:\n{context}\n\nAnswer:"""
    
    response = llm.invoke(prompt.format(query=query, context=context))#versity of data types for data mining
    output={'answer': response.content, 'sources': sources, 'confidence': confidence}
    if return_context:
        output['context']=context
    return output

In [96]:
result=rag_advanced("What programming language should I learn first for data mining??", rag_retriever, llm,top_k=3,min_score=0.1,return_context=True)
print("Answer:")
result['answer']

Retrieving documents for query: 'What programming language should I learn first for data mining??'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Answer:


'The provided documents do not contain sufficient information to answer this question.'